# Music 해상도(visual token) 실험

MMMU validation **Music 30문항**을 이미지 토큰 수만 바꿔서 다시 풉니다.
프롬프트, sampling(seed 42), 파서, judge, max_new_tokens(32768)는 **baseline(Run B)과 같고 해상도만 다릅니다.**

| 단계 (Qwen3-VL 이미지당 토큰) | min_pixels = max_pixels | 같은 값의 28² 표기 |
|---|---|---|
| **441** | 441 × 32² = 451,584 px | 576 × 28² |
| **784** | 784 × 32² = 802,816 px | 1024 × 28² |
| **1,568** | 1,568 × 32² = 1,605,632 px | 2048 × 28² |
| **3,920** | 3,920 × 32² = 4,014,080 px | 5120 × 28² |
| (참고) baseline | min 1,003,520 / max 4,014,080 px | 1280×28² ~ 5120×28² → Music 이미지 중앙값 약 1,034토큰 |

- Qwen3-VL은 패치 16px을 2×2로 묶어 **32×32픽셀당 이미지 토큰 1개**입니다.
- **min = max**로 둬야 모든 이미지가 그 토큰 수로 맞춰집니다(가로세로 비율은 유지, 32배수로 반올림되어 조금 차이 날 수 있음).
- 평가 코드는 Drive의 기존 `mmmu_project.zip`을 **수정 없이** 불러다 쓰고, 해상도 값만 바꿔서 실행합니다.
- 결과는 `MyDrive/mmmu_project/results/music_tok_<토큰>/`에 저장됩니다. 끊기면 같은 셀을 다시 실행하면 끝난 문항은 건너뜁니다.

## 1. GPU 확인 (A100 필요)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. 코드 불러오기 (Drive의 기존 zip 그대로)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/mmmu
!unzip -o -q /content/drive/MyDrive/mmmu_project.zip -d /content/mmmu
!ls /content/mmmu/mmmu_eval

## 3. 설치 — 끝나면 **커널 재시작** 후 4번부터 실행
(같은 런타임에서 이미 설치했다면 건너뛰어도 됩니다)

In [ ]:
!pip install -q -r /content/mmmu/requirements.txt
!pip uninstall -y -q torchaudio   # Colab 기본 torchaudio(CUDA 12.8)가 vLLM의 torch(CUDA 13.0)와 충돌
!python -c "import torch, vllm; print('torch', torch.__version__, '| vllm', vllm.__version__)"

## 4. 설정

In [ ]:
import os, sys
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # Colab Secrets에 등록. 코드에 토큰을 직접 쓰지 말 것

MODEL     = 'Qwen/Qwen3-VL-4B-Instruct'
REVISION  = 'ebb281ec70b05090aa6165b016eac8ec08e71b17'
JUDGE     = 'meta-llama/Llama-3.1-8B-Instruct'
JUDGE_REV = '0e9e39f249a16976918f6564b8830bc894c89659'

DATA_ROOT = '/content/hf_mmmu'
OUT_ROOT  = '/content/drive/MyDrive/mmmu_project/results'
BASELINE  = f'{OUT_ROOT}/baseline_mmmu_pro_cot'          # Run B (비교 기준)

TOKENS = [441, 784, 1568, 3920]                         # 이미지당 visual token
PIXELS = {t: t * 32 * 32 for t in TOKENS}               # min_pixels = max_pixels

# 3,920토큰 단계에서는 이미지 여러 장 문항의 프롬프트가 8,192토큰을 넘으므로 65536 (출력에는 영향 없음)
MAX_MODEL_LEN = 65536

sys.path.insert(0, '/content/mmmu/mmmu_eval')
import run_mmmu_eval as R                               # 기존 평가 코드 그대로 사용
print(PIXELS)

## 5. 서버 실행 도우미

In [ ]:
import subprocess, signal, time, requests
from argparse import Namespace

def start_server(cmd, port, log_path):
    log = open(log_path, 'w')
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
                            env={**os.environ, 'VLLM_USE_V2_MODEL_RUNNER': '0'})
    for _ in range(480):                                # 최대 40분 (첫 실행은 모델 다운로드 포함)
        try:
            if requests.get(f'http://localhost:{port}/v1/models', timeout=5).ok:
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            print(open(log_path).read()[-3000:])
            raise RuntimeError('vLLM 서버가 종료됐습니다. 위 로그를 확인하세요.')
        time.sleep(5)
    stop_server(proc)
    raise RuntimeError('vLLM 서버가 제시간에 준비되지 않았습니다.')

def stop_server(proc):
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(timeout=120)
    except Exception:
        os.killpg(proc.pid, signal.SIGKILL)
    time.sleep(10)                                      # GPU 메모리 해제 대기

## 6. 추론 — 단계마다 Qwen 서버를 새로 띄움
같은 이미지가 이전 단계 해상도로 처리된 캐시를 재사용하지 않도록 단계마다 서버를 다시 시작합니다(단계당 기동 1~3분).

In [ ]:
import json

for t in TOKENS:
    out = f'{OUT_ROOT}/music_tok_{t}'
    os.makedirs(out, exist_ok=True)
    print(f'\n========== Music @ {t} tokens/image ({PIXELS[t]:,} px) ==========')

    # 기존 코드의 해상도 상수만 이 단계 값으로 바꿈 (call_model이 이 값을 mm_processor_kwargs로 보냄)
    R.MIN_PIXELS = R.MAX_PIXELS = PIXELS[t]

    server = start_server(
        ['vllm', 'serve', MODEL, '--revision', REVISION, '--served-model-name', 'qwen3-vl-4b',
         '--dtype', 'bfloat16', '--seed', '42', '--max-model-len', str(MAX_MODEL_LEN),
         '--gpu-memory-utilization', '0.90', '--port', '8000'],
        8000, f'{out}/vllm_infer.log')
    try:
        R.cmd_infer(Namespace(
            output_dir=out, data_root=DATA_ROOT, subject='Music', all=False, limit=None,
            prompt='mmmu_pro_cot', max_new_tokens=32768, concurrency=32, request_timeout=3600,
            api_base='http://localhost:8000/v1', served_model_name='qwen3-vl-4b',
            model_path=MODEL, model_revision=REVISION))
    finally:
        stop_server(server)

    with open(f'{out}/resolution.json', 'w') as f:
        json.dump({'tokens_per_image': t, 'min_pixels': PIXELS[t], 'max_pixels': PIXELS[t],
                   'max_model_len': MAX_MODEL_LEN}, f, indent=2)

## 7. judge + 채점 — judge 서버는 한 번만 띄워 네 단계를 모두 판정

In [ ]:
judge = start_server(
    ['vllm', 'serve', JUDGE, '--revision', JUDGE_REV, '--served-model-name', 'judge',
     '--dtype', 'bfloat16', '--seed', '42', '--max-model-len', '20480',
     '--gpu-memory-utilization', '0.90', '--port', '8001'],
    8001, f'{OUT_ROOT}/music_tok_judge.log')
try:
    for t in TOKENS:
        R.cmd_judge(Namespace(output_dir=f'{OUT_ROOT}/music_tok_{t}', subject='Music', all=False, limit=None,
                              judge_api_base='http://localhost:8001/v1', judge_model='judge',
                              judge_revision=JUDGE_REV, concurrency=16))
finally:
    stop_server(judge)

for t in TOKENS:
    R.MIN_PIXELS = R.MAX_PIXELS = PIXELS[t]             # summary.json에 이 단계 해상도가 기록되도록
    R.cmd_score(Namespace(output_dir=f'{OUT_ROOT}/music_tok_{t}', subject='Music', all=False, limit=None))

## 8. 결과 비교
- **이미지 토큰 중앙값**으로 해상도가 실제로 적용됐는지 확인합니다(단계별로 뚜렷하게 달라야 정상).
- Music 30문항은 샘플링 노이즈가 커서 ±몇 문항 차이는 우연일 수 있습니다. 잘림 수와 끝까지 답한 문항의 정답을 함께 보세요.

In [ ]:
import json, os

def load(out_dir):
    rows = [json.loads(l) for l in open(f'{out_dir}/final_predictions.jsonl')]
    rows = [r for r in rows if r['subject'] == 'Music']
    done = [r for r in rows if r['finish_reason'] != 'length']
    pt = sorted(r['usage'].get('prompt_tokens', 0) for r in rows)
    return {
        'n': len(rows),
        'correct': sum(r['correct'] for r in rows),
        'truncated': sum(r['finish_reason'] == 'length' for r in rows),
        'errors': sum(r['finish_reason'] == 'error' for r in rows),
        'done': f"{sum(r['correct'] for r in done)}/{len(done)}",
        'prompt_tok': pt[len(pt) // 2] if pt else 0,
        'ok_ids': {r['id'] for r in rows if r['correct']},
    }

table = {'baseline (약 1,034)': load(BASELINE)}
for t in TOKENS:
    d = f'{OUT_ROOT}/music_tok_{t}'
    if os.path.exists(f'{d}/final_predictions.jsonl'):
        table[f'{t} tokens'] = load(d)

print(f"{'설정':<20}{'정답':>8}{'잘림':>6}{'오류':>6}{'끝까지 답한 문항 정답':>14}{'프롬프트 토큰 중앙값':>14}")
for k, v in table.items():
    print(f"{k:<20}{v['correct']:>5}/{v['n']:<2}{v['truncated']:>6}{v['errors']:>6}{v['done']:>14}{v['prompt_tok']:>14}")

base = table['baseline (약 1,034)']['ok_ids']
for k, v in table.items():
    if k.startswith('baseline'):
        continue
    gained = sorted(int(i.split('_')[-1]) for i in v['ok_ids'] - base)
    lost = sorted(int(i.split('_')[-1]) for i in base - v['ok_ids'])
    print(f'\n{k}: baseline 대비 새로 맞힘 {gained} / 놓침 {lost}')